# Approach A4: PrahokBART Dual Encoder (Khmer-Native Backbone)
## Khmer Legal Information Retrieval — Deep Learning Final Project

This notebook trains, tunes, and evaluates **Approach A4** on the official Cambodian Civil Code (2007) and Criminal Code (2009):
- **Backbone**: `nict-astrec-att/prahokbart_base` (35.8M parameters, 512-dim embeddings, SentencePiece 32,004 vocab)
- **Architecture**: Siamese dual tower with masked mean pooling and $L_2$ unit normalization
- **Loss**: InfoNCE with in-batch negatives ($\tau = 0.05$)
- **Optimizer**: AdamW + 10% linear warmup + Cosine Annealing decay
- **Benchmarks**: Primary T-Q (200 human-verified questions) and Secondary T-T (148 article titles) with 95% bootstrap CIs

### Step 1: Check GPU Acceleration
Verify that the Colab runtime is active (GPU or CPU).

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

### Step 2: Clone Repository & Install Dependencies

In [ ]:
!git clone https://github.com/mengchheanglong/khmer-legal-retrieval.git
%cd khmer-legal-retrieval
!git checkout feat/a4-prahokbart-encoder
!pip install -q -r requirements.txt
!pip install -q sentencepiece

### Step 3: Run Approach A4 Unit Tests

In [ ]:
!pytest tests/unit/test_a4_prahokbart.py -v

### Step 4: Execute Hyperparameter Grid Search
Grid: Learning Rate $\in \{5\times 10^{-5}, 1\times 10^{-4}\} \times \text{Weight Decay} = 0.01$.

In [ ]:
!python -m src.dl.experiments.tune_a4 --epochs 3 --batch-size 32 --patience 2

### Step 5: Display Hyperparameter Tuning Summary Table

In [ ]:
import pandas as pd
df_a4 = pd.read_csv('results/tuning/a4.csv')
print(df_a4.to_string(index=False))

### Step 6: Plot Training Dynamics
Plot per-epoch training loss and validation MRR@10 curves.

In [ ]:
import glob
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for f in sorted(glob.glob('results/logs/a4_*.csv')):
    name = f.split('/')[-1].replace('.csv', '').replace('a4_', '')
    df_run = pd.read_csv(f)
    ax1.plot(df_run['epoch'], df_run['train_loss'], marker='o', label=name)
    ax2.plot(df_run['epoch'], df_run['val_mrr10'], marker='s', label=name)

ax1.set_title('Training Loss (InfoNCE)')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.grid(True, linestyle='--', alpha=0.6)
ax1.legend()

ax2.set_title('Validation MRR@10')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MRR@10')
ax2.grid(True, linestyle='--', alpha=0.6)
ax2.legend()

plt.tight_layout()
plt.show()

### Step 7: View Final Benchmark Results on Full 1,976-Article Corpus

In [ ]:
import json
with open('results/metrics/a4_prahokbart.json', 'r', encoding='utf-8') as f:
    results = json.load(f)

print("=== Primary Benchmark (T-Q: 200 Questions) ===")
for metric, vals in results['tq_benchmark']['overall'].items():
    print(f"{metric:10s}: {vals['mean']:.4f} [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")

print("\n=== Secondary Benchmark (T-T: 148 Titles) ===")
for metric, vals in results['tt_benchmark']['overall'].items():
    print(f"{metric:10s}: {vals['mean']:.4f} [{vals['ci_lower']:.4f}, {vals['ci_upper']:.4f}]")